# Unified RL-Assisted MPC Combined Supervisor

This notebook runs the canonical polymer four-agent combined workflow using `Polymer/Data` and `Polymer/Results`.

In [ ]:
from pathlib import Path
import os

from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_notebook_summary

# Shared run controls.
RUN_MODE = "nominal"  # "nominal" | "disturb"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Enable/disable each agent block independently.
ENABLE_HORIZON = True
HORIZON_STATE_MODE = "standard"

ENABLE_MATRIX = True
MATRIX_AGENT_KIND = "td3"  # "td3" | "sac"
MATRIX_STATE_MODE = "standard"

ENABLE_WEIGHTS = True
WEIGHTS_AGENT_KIND = "td3"  # "td3" | "sac"
WEIGHTS_STATE_MODE = "standard"

ENABLE_RESIDUAL = True
RESIDUAL_AGENT_KIND = "td3"  # "td3" | "sac"
RESIDUAL_STATE_MODE = "standard"
USE_RHO_AUTHORITY = True

# Optional directory overrides. Leave as None to use Polymer/Data and Polymer/Results.
POLYMER_DATA_DIR_OVERRIDE = None
POLYMER_RESULTS_DIR_OVERRIDE = None

# Optional naming/path overrides. Leave as None to use the canonical mode-based defaults.
RESULT_PREFIX_OVERRIDE = None
COMPARE_PREFIX_OVERRIDE = None
BASELINE_MPC_PATH_OVERRIDE = None

# Optional run-size overrides. Leave as None to use the defaults from RUN_PROFILE_MAP.
N_TESTS_OVERRIDE = None
SET_POINTS_LEN_OVERRIDE = None
WARM_START_OVERRIDE = None
TEST_CYCLE_OVERRIDE = None
PLOT_START_EPISODE_OVERRIDE = None
COMPARE_START_EPISODE_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

def build_combined_suffix() -> str:
    parts = []
    if ENABLE_HORIZON:
        parts.append(f"h_dqn_{HORIZON_STATE_MODE}")
    if ENABLE_MATRIX:
        parts.append(f"m_{MATRIX_AGENT_KIND}_{MATRIX_STATE_MODE}")
    if ENABLE_WEIGHTS:
        parts.append(f"w_{WEIGHTS_AGENT_KIND}_{WEIGHTS_STATE_MODE}")
    if ENABLE_RESIDUAL:
        rho_suffix = "rho" if USE_RHO_AUTHORITY else "no_rho"
        parts.append(f"r_{RESIDUAL_AGENT_KIND}_{RESIDUAL_STATE_MODE}_{rho_suffix}")
    return "__".join(parts) if parts else "baseline"

COMBINED_SUFFIX = build_combined_suffix()
RUN_PROFILE_MAP = {
    "nominal": {
        "baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "nominal", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "result_prefix": f"combined_nominal_{COMBINED_SUFFIX}",
        "compare_prefix": f"nominal_compare_combined_{COMBINED_SUFFIX}",
        "compare_mode": "nominal",
        "plot_start_episode": 2,
        "compare_start_episode": 2,
    },
    "disturb": {
        "baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "disturb", data_override=POLYMER_DATA_DIR_OVERRIDE),
        "result_prefix": f"combined_disturb_{COMBINED_SUFFIX}",
        "compare_prefix": f"disturb_compare_combined_{COMBINED_SUFFIX}",
        "compare_mode": "disturb",
        "plot_start_episode": 2,
        "compare_start_episode": 2,
    },
}
RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix"]
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix"]
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else RUN_PROFILE["baseline_mpc_path"]

print_notebook_summary(
    "Polymer combined configuration",
    {
        "Polymer data dir": DATA_DIR,
        "Polymer results": RESULT_DIR,
        "Run mode": RUN_MODE,
        "Enabled horizon": ENABLE_HORIZON,
        "Enabled matrix": ENABLE_MATRIX,
        "Enabled weights": ENABLE_WEIGHTS,
        "Enabled residual": ENABLE_RESIDUAL,
        "Baseline MPC": BASELINE_MPC_PATH,
    },
)


In [ ]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from systems.polymer import (
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.combined_runner import run_combined_supervisor
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.observer import compute_observer_gain
from utils.plotting import plot_combined_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim


In [ ]:
# Build the polymer plant and load the canonical polymer data bundle.
system_params = POLYMER_SYSTEM_PARAMS.copy()
system_design_params = POLYMER_DESIGN_PARAMS.copy()
system_steady_state_inputs = POLYMER_SS_INPUTS.copy()
delta_t = POLYMER_DELTA_T_HOURS

cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}

setpoint_y = POLYMER_SETPOINT_RANGE_PHYS.copy()
u_min = POLYMER_INPUT_BOUNDS["u_min"].copy()
u_max = POLYMER_INPUT_BOUNDS["u_max"].copy()
system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = 2
y_sp_scenario = POLYMER_RL_SETPOINTS_PHYS.copy()
y_sp_scenario = (
    apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)


In [ ]:
# User-editable experiment defaults.
# Set the *_OVERRIDE values in cell 2 if you want to change the run size without editing this cell.
DEFAULT_N_TESTS = 200
DEFAULT_SET_POINTS_LEN = 200
DEFAULT_WARM_START = 0
DEFAULT_TEST_CYCLE = [False, False, False, False, False]

n_tests = RUN_PROFILE.get("n_tests", DEFAULT_N_TESTS) if N_TESTS_OVERRIDE is None else int(N_TESTS_OVERRIDE)
set_points_len = RUN_PROFILE.get("set_points_len", DEFAULT_SET_POINTS_LEN) if SET_POINTS_LEN_OVERRIDE is None else int(SET_POINTS_LEN_OVERRIDE)
warm_start = RUN_PROFILE.get("warm_start", DEFAULT_WARM_START) if WARM_START_OVERRIDE is None else int(WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE.get("test_cycle", DEFAULT_TEST_CYCLE)) if TEST_CYCLE_OVERRIDE is None else list(TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else int(PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else int(COMPARE_START_EPISODE_OVERRIDE)

N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACTOR_FREEZE = 0
DECISION_INTERVAL = 4

PREDICT_GRID = HORIZON_PREDICT_GRID
CONTROL_GRID = HORIZON_CONTROL_GRID
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
predict_h = 9
cont_h = 3

Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
MODEL_LOW = np.array([0.95, 0.95, 0.95])
MODEL_HIGH = np.array([1.05, 1.05, 1.05])
WEIGHTS_LOW = np.array([0.9, 0.9, 0.9, 0.9])
WEIGHTS_HIGH = np.array([1.1, 1.1, 1.1, 1.1])
RESIDUAL_LOW = np.array([-0.5, -0.5])
RESIDUAL_HIGH = np.array([0.5, 0.5])

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    k_rel,
    band_floor_phys,
    Q_diag,
    R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)
print(reward_params)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85


def make_continuous_agent(agent_kind, state_dim, action_dim):
    if agent_kind == "td3":
        return TD3Agent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            batch_size=128,
            policy_delay=4,
            target_policy_smoothing_noise_std=0.1,
            noise_clip=0.2,
            max_action=1.0,
            tau=0.005,
            std_start=0.2,
            std_end=0.02,
            std_decay_rate=0.99995,
            std_decay_mode="exp",
            buffer_size=25_000,
            device=DEVICE,
            actor_freeze=ACTOR_FREEZE,
        )
    if agent_kind == "sac":
        return SACAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            alpha_lr=1e-4,
            batch_size=128,
            grad_clip_norm=10.0,
            init_alpha=0.01,
            learn_alpha=True,
            target_entropy=-action_dim,
            target_update="soft",
            tau=0.005,
            hard_update_interval=10_000,
            activation="relu",
            use_layernorm=False,
            dropout=0.0,
            max_action=1.0,
            buffer_size=20_000,
            use_per=True,
            device=DEVICE,
            use_adamw=True,
            actor_freeze=ACTOR_FREEZE,
        )
    raise ValueError("Continuous agent kind must be 'td3' or 'sac'.")


horizon_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, HORIZON_STATE_MODE)
matrix_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, MATRIX_STATE_MODE)
weights_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, WEIGHTS_STATE_MODE)
residual_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, RESIDUAL_STATE_MODE)

horizon_agent = None
if ENABLE_HORIZON:
    horizon_agent = DQNAgent(
        state_dim=horizon_state_dim,
        action_dim=len(HORIZON_RECIPES),
        hidden_dim=[512, 512, 512, 512, 512],
        gamma=0.99,
        lr=1e-4,
        batch_size=128,
        buffer_size=40_000,
        grad_clip_norm=10.0,
        double_dqn=True,
        target_update="soft",
        tau=0.01,
        hard_update_interval=10_000,
        target_combine="q1",
        activation="relu",
        use_layer_norm=False,
        dropout=0.0,
        device=DEVICE,
        eps_start=0.3,
        eps_end=0.01,
        eps_decay_rate=0.99999,
        eps_decay_mode="exp",
    )

matrix_agent = make_continuous_agent(MATRIX_AGENT_KIND, matrix_state_dim, 1 + N_INPUTS) if ENABLE_MATRIX else None
weights_agent = make_continuous_agent(WEIGHTS_AGENT_KIND, weights_state_dim, 4) if ENABLE_WEIGHTS else None
residual_agent = make_continuous_agent(RESIDUAL_AGENT_KIND, residual_state_dim, N_INPUTS) if ENABLE_RESIDUAL else None

print(f"Using device: {DEVICE}")
print(f"Enabled agents: horizon={ENABLE_HORIZON}, matrix={ENABLE_MATRIX}, weights={ENABLE_WEIGHTS}, residual={ENABLE_RESIDUAL}")
print(f"State dimensions | horizon={horizon_state_dim}, matrix={matrix_state_dim}, weights={weights_state_dim}, residual={residual_state_dim}")


print_notebook_summary(
    "Resolved experiment parameters",
    {
        "n_tests": n_tests,
        "set_points_len": set_points_len,
        "warm_start": warm_start,
        "plot_start_episode": PLOT_START_EPISODE,
        "compare_start_episode": COMPARE_START_EPISODE,
    },
)


In [ ]:
combined_cfg = {
    "run_mode": RUN_MODE,
    "decision_interval": DECISION_INTERVAL,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "horizon_cfg": {
        "enabled": ENABLE_HORIZON,
        "state_mode": HORIZON_STATE_MODE,
        "horizon_recipes": HORIZON_RECIPES,
        "default_horizons": (predict_h, cont_h),
    },
    "matrix_cfg": {
        "enabled": ENABLE_MATRIX,
        "agent_kind": MATRIX_AGENT_KIND,
        "state_mode": MATRIX_STATE_MODE,
        "low_coef": MODEL_LOW,
        "high_coef": MODEL_HIGH,
    },
    "weight_cfg": {
        "enabled": ENABLE_WEIGHTS,
        "agent_kind": WEIGHTS_AGENT_KIND,
        "state_mode": WEIGHTS_STATE_MODE,
        "low_coef": WEIGHTS_LOW,
        "high_coef": WEIGHTS_HIGH,
    },
    "residual_cfg": {
        "enabled": ENABLE_RESIDUAL,
        "agent_kind": RESIDUAL_AGENT_KIND,
        "state_mode": RESIDUAL_STATE_MODE,
        "use_rho_authority": USE_RHO_AUTHORITY,
        "low_coef": RESIDUAL_LOW,
        "high_coef": RESIDUAL_HIGH,
    },
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
agents = {}
if ENABLE_HORIZON:
    agents["horizon"] = horizon_agent
if ENABLE_MATRIX:
    agents["matrix"] = matrix_agent
if ENABLE_WEIGHTS:
    agents["weights"] = weights_agent
if ENABLE_RESIDUAL:
    agents["residual"] = residual_agent

runtime_ctx = {
    "system": cstr,
    "agents": agents,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": POLYMER_OBSERVER_POLES.copy(),
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_metadata": POLYMER_SYSTEM_METADATA,
}

result_bundle = run_combined_supervisor(combined_cfg=combined_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = RUN_PROFILE["baseline_mpc_path"]


In [ ]:
out_dir_rl = plot_combined_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "reward_fn": reward_fn,
        "system_metadata": POLYMER_SYSTEM_METADATA,
        "include_baseline_compare": True,
        "compare_mode": RUN_PROFILE["compare_mode"],
        "compare_prefix": COMPARE_PREFIX,
        "compare_directory": RESULT_DIR,
        "mpc_path_or_dir": BASELINE_MPC_PATH,
        "compare_start_episode": COMPARE_START_EPISODE,
    },
)

print("Combined result directory:", out_dir_rl)
